# Can we predict term-deposit subscription?

This notebook tests whether the `y` outcome can be predicted from the bank marketing data in `data.csv`. The primary metric is **average precision (AP)** because only about 12% of calls result in a subscription; ROC AUC is reported as a secondary ranking metric.

Two prediction settings are deliberately separated:

1. **Pre-call / actionable:** excludes `duration`, which is not known until the call is over. This is the honest model for deciding whom to call.
2. **Post-call / explanatory:** includes `duration`. It can estimate the outcome after a completed call, but it must not be used to claim pre-call performance.

The workflow uses a stratified train/test split, keeps the test set untouched until final evaluation, handles missing and unseen categories inside a pipeline, compares against a prevalence-only dummy model, and reports bootstrap confidence intervals.

In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score, classification_report,
    ConfusionMatrixDisplay, PrecisionRecallDisplay, RocCurveDisplay,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder

RANDOM_STATE = 42
pd.set_option('display.max_columns', 50)
warnings.filterwarnings('ignore', category=FutureWarning)

In [ ]:
DATA_PATH = Path('data.csv')
if not DATA_PATH.exists():
    raise FileNotFoundError(f'Expected {DATA_PATH.resolve()}')

df = pd.read_csv(DATA_PATH)
assert 'y' in df.columns, 'The target column y is missing.'
assert df['y'].notna().all(), 'This experiment expects a fully observed target.'
assert set(df['y'].unique()) <= {'yes', 'no'}, 'Unexpected target labels.'

print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} columns')
display(df.head())
display(pd.DataFrame({
    'dtype': df.dtypes.astype(str),
    'missing_n': df.isna().sum(),
    'missing_pct': (100 * df.isna().mean()).round(1),
    'unique': df.nunique(dropna=True),
}))
print(f'Duplicate rows: {df.duplicated().sum():,}')

In [ ]:
target_counts = df['y'].value_counts().rename_axis('y').to_frame('count')
target_counts['percent'] = (100 * target_counts['count'] / len(df)).round(2)
display(target_counts)

ax = target_counts['count'].plot.bar(color=['#64748b', '#0f766e'], rot=0)
ax.set(title='Target distribution', ylabel='Calls', xlabel='Subscribed')
plt.tight_layout()
plt.show()

## Experimental design

A fixed 20% stratified test set provides the final estimate. Five-fold stratified cross-validation on the remaining 80% measures variation during model development. The gradient-boosted tree model captures nonlinearities and interactions without requiring linear assumptions.

Missing categorical values are explicitly imputed as `missing`; unknown categories at prediction time are safely encoded as missing. All preprocessing is fitted only on training folds, preventing preprocessing leakage.

In [ ]:
X = df.drop(columns='y')
y = df['y'].eq('yes').astype('int8')
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
)
print(f'Train: {len(X_train):,}; test: {len(X_test):,}')
print(f'Positive rate — train: {y_train.mean():.3%}; test: {y_test.mean():.3%}')

def make_model(feature_frame):
    categorical = feature_frame.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    numeric = feature_frame.columns.difference(categorical, sort=False).tolist()

    categorical_pipe = Pipeline([
        ('impute', SimpleImputer(strategy='constant', fill_value='missing')),
        ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ])
    preprocess = ColumnTransformer([
        ('categorical', categorical_pipe, categorical),
        ('numeric', SimpleImputer(strategy='median'), numeric),
    ], verbose_feature_names_out=False)

    # The transformer emits categorical columns first. Negative codes are treated as missing.
    categorical_mask = [True] * len(categorical) + [False] * len(numeric)
    model = HistGradientBoostingClassifier(
        learning_rate=0.08, max_iter=220, max_leaf_nodes=31,
        min_samples_leaf=30, l2_regularization=1.0,
        categorical_features=categorical_mask, random_state=RANDOM_STATE,
    )
    return Pipeline([('preprocess', preprocess), ('model', model)])

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {'average_precision': 'average_precision', 'roc_auc': 'roc_auc'}
scenarios = {
    'Pre-call (no duration)': [c for c in X.columns if c != 'duration'],
    'Post-call (with duration)': X.columns.tolist(),
}

cv_rows = []
models = {}
test_predictions = {}
for name, columns in scenarios.items():
    estimator = make_model(X_train[columns])
    scores = cross_validate(
        estimator, X_train[columns], y_train, cv=cv, scoring=scoring, n_jobs=-1
    )
    cv_rows.append({
        'scenario': name,
        'CV AP mean': scores['test_average_precision'].mean(),
        'CV AP sd': scores['test_average_precision'].std(),
        'CV ROC AUC mean': scores['test_roc_auc'].mean(),
        'CV ROC AUC sd': scores['test_roc_auc'].std(),
    })
    estimator.fit(X_train[columns], y_train)
    models[name] = estimator
    test_predictions[name] = estimator.predict_proba(X_test[columns])[:, 1]

cv_results = pd.DataFrame(cv_rows).set_index('scenario')
display(cv_results.style.format('{:.3f}'))

In [ ]:
dummy = DummyClassifier(strategy='prior').fit(np.zeros((len(y_train), 1)), y_train)
dummy_score = dummy.predict_proba(np.zeros((len(y_test), 1)))[:, 1]
all_predictions = {'Prevalence-only baseline': dummy_score, **test_predictions}

test_rows = []
for name, probabilities in all_predictions.items():
    test_rows.append({
        'scenario': name,
        'Test average precision': average_precision_score(y_test, probabilities),
        'Test ROC AUC': roc_auc_score(y_test, probabilities),
    })
test_results = pd.DataFrame(test_rows).set_index('scenario')
display(test_results.style.format('{:.3f}'))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for name, probabilities in all_predictions.items():
    PrecisionRecallDisplay.from_predictions(y_test, probabilities, name=name, ax=axes[0])
    RocCurveDisplay.from_predictions(y_test, probabilities, name=name, ax=axes[1])
axes[0].set_title('Precision–recall curves (held-out test set)')
axes[1].set_title('ROC curves (held-out test set)')
plt.tight_layout()
plt.show()

## Uncertainty on the held-out test set

The percentile bootstrap below resamples test rows 1,000 times. This describes sampling uncertainty for this particular split; it does not cover future population drift or campaign changes.

In [ ]:
def bootstrap_metric_ci(y_true, score, metric, n_boot=1000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    y_array = np.asarray(y_true)
    score_array = np.asarray(score)
    values = []
    for _ in range(n_boot):
        idx = rng.integers(0, len(y_array), len(y_array))
        if np.unique(y_array[idx]).size == 2:
            values.append(metric(y_array[idx], score_array[idx]))
    return np.percentile(values, [2.5, 50, 97.5])

ci_rows = []
for name, probabilities in test_predictions.items():
    ap_ci = bootstrap_metric_ci(y_test, probabilities, average_precision_score)
    auc_ci = bootstrap_metric_ci(y_test, probabilities, roc_auc_score)
    ci_rows.append({
        'scenario': name,
        'AP median [95% CI]': f'{ap_ci[1]:.3f} [{ap_ci[0]:.3f}, {ap_ci[2]:.3f}]',
        'ROC AUC median [95% CI]': f'{auc_ci[1]:.3f} [{auc_ci[0]:.3f}, {auc_ci[2]:.3f}]',
    })
display(pd.DataFrame(ci_rows).set_index('scenario'))

## What drives the actionable model?

Permutation importance measures the drop in test-set AP after shuffling one original feature. It is model-specific and predictive—not causal. Correlated features may split importance.

In [ ]:
pre_name = 'Pre-call (no duration)'
pre_columns = scenarios[pre_name]
importance = permutation_importance(
    models[pre_name], X_test[pre_columns], y_test,
    scoring='average_precision', n_repeats=5, random_state=RANDOM_STATE, n_jobs=-1,
)
importance_df = (pd.DataFrame({
    'feature': pre_columns,
    'AP decrease mean': importance.importances_mean,
    'AP decrease sd': importance.importances_std,
}).sort_values('AP decrease mean', ascending=False).reset_index(drop=True))
display(importance_df.head(12).style.format({'AP decrease mean': '{:.4f}', 'AP decrease sd': '{:.4f}'}))

plot_df = importance_df.head(12).sort_values('AP decrease mean')
plt.figure(figsize=(8, 5))
plt.barh(plot_df['feature'], plot_df['AP decrease mean'], xerr=plot_df['AP decrease sd'], color='#0f766e')
plt.xlabel('Decrease in average precision after shuffling')
plt.title('Pre-call model: permutation importance on test set')
plt.tight_layout()
plt.show()

## Operational view: targeting the highest-scored clients

A ranking model is often more useful than a fixed 0.5 threshold. The table shows how many actual subscribers are captured if the campaign calls only the top-scored fraction of the held-out clients. `Lift` compares precision in that group with calling clients at random.

In [ ]:
def targeting_table(y_true, score, fractions=(0.05, 0.10, 0.20, 0.30, 0.50)):
    ranked = pd.DataFrame({'y': np.asarray(y_true), 'score': score}).sort_values('score', ascending=False)
    rows = []
    for fraction in fractions:
        n = int(np.ceil(len(ranked) * fraction))
        selected = ranked.head(n)
        precision = selected['y'].mean()
        rows.append({
            'Top fraction called': fraction,
            'Clients called': n,
            'Precision': precision,
            'Recall / subscribers captured': selected['y'].sum() / ranked['y'].sum(),
            'Lift vs random': precision / ranked['y'].mean(),
        })
    return pd.DataFrame(rows)

targeting = targeting_table(y_test, test_predictions[pre_name])
display(targeting.style.format({
    'Top fraction called': '{:.0%}', 'Precision': '{:.1%}',
    'Recall / subscribers captured': '{:.1%}', 'Lift vs random': '{:.2f}×',
}))

## Conclusion

The next cell generates a compact, data-dependent conclusion. Whether the model is useful in production still depends on campaign economics, probability calibration, fairness review, and validation on a later campaign. A random split estimates performance on similar calls, not necessarily on future time periods.

In [ ]:
pre_ap = test_results.loc[pre_name, 'Test average precision']
pre_auc = test_results.loc[pre_name, 'Test ROC AUC']
post_name = 'Post-call (with duration)'
post_ap = test_results.loc[post_name, 'Test average precision']
baseline_ap = y_test.mean()
top10 = targeting.loc[targeting['Top fraction called'].eq(0.10)].iloc[0]

print(
    f'Yes—there is useful predictive signal before the call. On the untouched test set, '
    f'the actionable model achieved AP={pre_ap:.3f} versus a prevalence baseline of '
    f'{baseline_ap:.3f}, and ROC AUC={pre_auc:.3f}. Calling the top 10% by score '
    f'would capture {top10["Recall / subscribers captured"]:.1%} of subscribers at '
    f'{top10["Lift vs random"]:.2f}× the random-call precision. '
    f'Including call duration raises AP to {post_ap:.3f}, but that is post-call information '
    'and therefore unsuitable for deciding whom to call. Before deployment, validate on a '
    'strictly later campaign and choose the targeting cutoff from contact cost and deposit value.'
)